# Project 02 – StyleGAN2 Face Generation
Progressive training: **256 → 512 → 1024**

**Setup flow**
1. `git clone` → code (always up-to-date with repo)
2. Google Drive mount → data zip lives here; checkpoints backed up here
3. First run: unzip data → `/content/ffhq/`  
   **Re-run: skip unzip, read images directly from Drive** (`USE_DRIVE_IMAGES=True`)
4. WandB login → train

## 1. Install packages + clone repo

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/jyun-chae/skku-2-openai_pa2.git'
REPO_BRANCH = 'main'
# ────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

# ── Packages ─────────────────────────────────────────────────────────────────
!pip install -q wandb pytorch-fid pyyaml scikit-image onnx
!pip install -q 'torch>=2.2.0' torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade

# ── Clone or pull ─────────────────────────────────────────────────────────────
REPO_DIR = '/content/project02'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', REPO_BRANCH], check=True)
    print('Pulled latest.')
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
        check=True,
    )
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print(f'torch={torch.__version__}  CUDA={torch.cuda.is_available()}',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── USER CONFIG ──────────────────────────────────────────────────────────────
DRIVE_DIR      = '/content/drive/MyDrive/project02'
DRIVE_DATA_DIR = f'{DRIVE_DIR}/data'           # where your zip files live
DRIVE_CKPT_DIR = f'{DRIVE_DIR}/checkpoints'    # checkpoint backup
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Drive ready.')

## 3. Data setup

**First run** (`USE_DRIVE_IMAGES = False`): unzip from Drive → `/content/ffhq/`  
**Re-run** (`USE_DRIVE_IMAGES = True`): images already extracted to Drive → read directly (no unzip)

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
USE_DRIVE_IMAGES = False   # True → images already extracted on Drive; skip unzip

# zip filenames in DRIVE_DATA_DIR  (e.g. train_50k_256.zip / valid_10k_256.zip)
TRAIN_ZIP  = f'{DRIVE_DATA_DIR}/train_50k_256.zip'
VALID_ZIP  = f'{DRIVE_DATA_DIR}/valid_10k_256.zip'
EXTRACT_TO = '/content/ffhq'

# If USE_DRIVE_IMAGES=True: point to already-extracted dirs on Drive
DRIVE_TRAIN_DIR = f'{DRIVE_DATA_DIR}/train'
DRIVE_VALID_DIR = f'{DRIVE_DATA_DIR}/valid'
# ────────────────────────────────────────────────────────────────────────────

from pathlib import Path
from parallel_unzip import parallel_unzip

if USE_DRIVE_IMAGES:
    TRAIN_ROOT = DRIVE_TRAIN_DIR
    VALID_ROOT = DRIVE_VALID_DIR
    print('Using Drive images directly (no unzip).')
else:
    train_dir = Path(EXTRACT_TO) / 'train'
    valid_dir = Path(EXTRACT_TO) / 'valid'

    if not train_dir.exists() or len(list(train_dir.iterdir())) < 100:
        print('Extracting train zip…')
        parallel_unzip(TRAIN_ZIP, train_dir, strip_dirs=True, stage_local=True)
    if not valid_dir.exists() or len(list(valid_dir.iterdir())) < 100:
        print('Extracting valid zip…')
        parallel_unzip(VALID_ZIP, valid_dir, strip_dirs=True, stage_local=True)

    TRAIN_ROOT = str(train_dir)
    VALID_ROOT = str(valid_dir)

print(f'TRAIN_ROOT = {TRAIN_ROOT}  ({len(list(Path(TRAIN_ROOT).iterdir()))} imgs)')
print(f'VALID_ROOT = {VALID_ROOT}  ({len(list(Path(VALID_ROOT).iterdir()))} imgs)')

## 4. WandB login

In [ ]:
import wandb

# ── USER CONFIG ──────────────────────────────────────────────────────────────
WANDB_API_KEY = 'YOUR_WANDB_API_KEY_HERE'
WANDB_PROJECT = 'project02-stylegan2-256'
WANDB_ENTITY  = None   # your WandB username/org, or None
# ────────────────────────────────────────────────────────────────────────────

wandb.login(key=WANDB_API_KEY)
print('WandB logged in.')

## 5. Config helper + shared setup

In [ ]:
import yaml
from types import SimpleNamespace
from src.models.generator import StyleGAN2Generator
from src.models.discriminator import StyleGAN2Discriminator
from src.training.trainer import Trainer
from src.data.dataset import build_dataloader
from src.utils.fid_score import ValidFIDCache

def load_cfg(yaml_path):
    with open(yaml_path) as f:
        return SimpleNamespace(**yaml.safe_load(f))

CFG_DIR  = f'{REPO_DIR}/configs'
CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
device = torch.device('cuda')

# Sanity-check parameter count
G_check = StyleGAN2Generator(resolution=1024)
print(f'G params: {G_check.count_parameters():,} ({G_check.count_parameters()/1e6:.3f}M)  [limit: 40M]')
assert G_check.count_parameters() < 40_000_000
del G_check

## 6. Train – Stage 1: 256×256
Train from scratch. Estimated ~6–8 h on A100.

In [ ]:
cfg256 = load_cfg(f'{CFG_DIR}/train_256.yaml')

# split=None → load all images in each pre-split directory
train_loader = build_dataloader(TRAIN_ROOT, None, cfg256.resolution, cfg256.batch_size, cfg256.num_workers, aug=True)
valid_loader = build_dataloader(VALID_ROOT, None, cfg256.resolution, cfg256.batch_size, cfg256.num_workers, aug=False)

run256 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                    name='train-256', config=vars(cfg256), resume='allow')

trainer256 = Trainer(cfg256)
# trainer256.load(f'{CKPT_DIR}/ckpt_256_XXXXXXX.pth')  # ← resume from checkpoint

trainer256.fit(train_loader, valid_loader,
               wandb_run=run256,
               drive_backup_dir=DRIVE_CKPT_DIR,
               ckpt_dir=CKPT_DIR)
run256.finish()
CKPT_256 = f'{CKPT_DIR}/ckpt_256_final.pth'
print('Stage 1 done →', CKPT_256)

## 7. Train – Stage 2: 512×512
Fine-tune from 256 checkpoint. Estimated ~5–8 h on A100.

In [ ]:
cfg512 = load_cfg(f'{CFG_DIR}/train_512.yaml')

train_loader_512 = build_dataloader(TRAIN_ROOT, None, cfg512.resolution, cfg512.batch_size, cfg512.num_workers, aug=True)
valid_loader_512 = build_dataloader(VALID_ROOT, None, cfg512.resolution, cfg512.batch_size, cfg512.num_workers, aug=False)

run512 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                    name='train-512', config=vars(cfg512), resume='allow')

trainer512 = Trainer(cfg512)

# ── USER CONFIG ──────────────────────────────────────────────────────────────
CKPT_256 = f'{CKPT_DIR}/ckpt_256_final.pth'
# ────────────────────────────────────────────────────────────────────────────
state = torch.load(CKPT_256, map_location=device)
trainer512.G.load_from_lower_resolution(state['G'])
trainer512.D.load_state_dict(state['D'], strict=False)

trainer512.fit(train_loader_512, valid_loader_512,
               wandb_run=run512,
               drive_backup_dir=DRIVE_CKPT_DIR,
               ckpt_dir=CKPT_DIR)
run512.finish()
CKPT_512 = f'{CKPT_DIR}/ckpt_512_final.pth'
print('Stage 2 done →', CKPT_512)

## 8. Train – Stage 3: 1024×1024
Fine-tune from 512 checkpoint. Estimated ~15–25 h on A100.

In [ ]:
cfg1024 = load_cfg(f'{CFG_DIR}/train_1024.yaml')

train_loader_1024 = build_dataloader(TRAIN_ROOT, None, cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers, aug=True)
valid_loader_1024 = build_dataloader(VALID_ROOT, None, cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers, aug=False)

run1024 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name='train-1024', config=vars(cfg1024), resume='allow')

trainer1024 = Trainer(cfg1024)

# ── USER CONFIG ──────────────────────────────────────────────────────────────
CKPT_512 = f'{CKPT_DIR}/ckpt_512_final.pth'
# ────────────────────────────────────────────────────────────────────────────
state = torch.load(CKPT_512, map_location=device)
trainer1024.G.load_from_lower_resolution(state['G'])
trainer1024.D.load_state_dict(state['D'], strict=False)

trainer1024.fit(train_loader_1024, valid_loader_1024,
                wandb_run=run1024,
                drive_backup_dir=DRIVE_CKPT_DIR,
                ckpt_dir=CKPT_DIR)
run1024.finish()
CKPT_1024 = f'{CKPT_DIR}/ckpt_1024_final.pth'
print('Stage 3 done →', CKPT_1024)

## 9. FID evaluation (valid set)

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
EVAL_CKPT = CKPT_1024
EVAL_RES  = 1024
# ────────────────────────────────────────────────────────────────────────────

cfg_eval = load_cfg(f'{CFG_DIR}/train_{EVAL_RES}.yaml')
G_eval = StyleGAN2Generator(
    resolution=cfg_eval.resolution, z_dim=cfg_eval.z_dim,
    w_dim=cfg_eval.w_dim, channel_base=cfg_eval.channel_base,
    channel_max=cfg_eval.channel_max, mapping_layers=cfg_eval.mapping_layers,
).to(device)
G_eval.load_state_dict(torch.load(EVAL_CKPT, map_location=device)['G'])
G_eval.eval()
print(f'G params: {G_eval.count_parameters():,}')

valid_loader_eval = build_dataloader(VALID_ROOT, None, EVAL_RES, 8, 4, aug=False)
fid_cache = ValidFIDCache(valid_loader_eval, device)
fid = fid_cache.compute(G_eval, n_gen=10000, batch_size=16)
print(f'FID (valid, 10k): {fid:.2f}')

## 10. Generate samples

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

mean_w = G_eval.mapping.mean_latent(n_samples=4096, device=str(device))
with torch.no_grad():
    z = torch.randn(16, cfg_eval.z_dim, device=device)
    imgs = G_eval(z, noise_mode='const', truncation=0.7, mean_w=mean_w)
    imgs = (imgs.clamp(-1, 1) + 1) / 2

grid = make_grid(imgs.cpu(), nrow=4, padding=2)
plt.figure(figsize=(14, 14))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title(f'StyleGAN2 {EVAL_RES}×{EVAL_RES}  truncation=0.7')
plt.tight_layout()
plt.savefig('/content/samples.png', dpi=150)
plt.show()

## 11. Export ONNX (submission)

In [ ]:
import shutil
ONNX_PATH = f'/content/generator_{EVAL_RES}.onnx'
onnx_params = G_eval.export_onnx(ONNX_PATH, batch_size=1)
shutil.copy(ONNX_PATH, f'{DRIVE_DIR}/generator_{EVAL_RES}.onnx')
print(f'Saved to Drive: {DRIVE_DIR}/generator_{EVAL_RES}.onnx')